# Sentiment Analysis on Merged Comments

This notebook performs sentiment analysis on comments grouped by post ID using the Cardiff NLP Twitter RoBERTa model. It provides options to upload files directly or mount Google Drive.

## Features:
- GPU acceleration support
- File upload or Google Drive mounting
- Sentiment analysis using Hugging Face transformers
- Results export to CSV

## 1. Setup Environment

Check GPU availability and set up the device for sentiment analysis.

In [ ]:
import torch
import pandas as pd
from transformers import pipeline
import os

# Check device availability
device = 0 if torch.cuda.is_available() else -1
print(f"Device set to use {'CUDA (GPU)' if device == 0 else 'CPU'}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## 2. Upload File or Mount Google Drive

Choose one of the following options to access your data file:

### Option A: Upload File Directly

In [ ]:
from google.colab import files

# Upload your CSV file
print("Please upload your preprocessed_data.csv file:")
uploaded = files.upload()

# Get the filename
filename = list(uploaded.keys())[0]
print(f"Uploaded file: {filename}")

# Set the file path
csv_path = filename

### Option B: Mount Google Drive (Alternative)

Uncomment and run the cell below if you prefer to use Google Drive:

In [ ]:
# Uncomment the lines below if you want to use Google Drive instead

# from google.colab import drive
# drive.mount('/content/drive')

# # Set the path to your file in Google Drive
# csv_path = '/content/drive/MyDrive/path/to/your/preprocessed_data.csv'
# print(f"Using file from Google Drive: {csv_path}")

## 3. Install Required Libraries

Install the necessary libraries for sentiment analysis.

In [ ]:
# Install required libraries
!pip install transformers torch pandas

print("✅ Libraries installed successfully!")

## 4. Load Dataset

Load the CSV file and inspect the data structure.

In [ ]:
# Load the dataset
try:
    df = pd.read_csv(csv_path)
    print(f"✅ Dataset loaded successfully!")
    print(f"Shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    print("\nFirst few rows:")
    print(df.head())
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    print("Please check the file path and format.")

## 5. Group Comments by Post ID

Merge all comment texts for each post into a single string for sentiment analysis.

In [ ]:
# Group all comment texts by post_id
grouped = df.groupby('post_id')['comment_text'].apply(
    lambda comments: ' '.join(str(c) for c in comments if pd.notna(c))
)

print(f"✅ Comments grouped successfully!")
print(f"Number of posts: {len(grouped)}")
print("\nMerged comment text for sentiment analysis (first few rows):")
print(grouped.head())

# Check text lengths to ensure they don't exceed model limits
text_lengths = grouped.str.len()
print(f"\nText length statistics:")
print(f"Mean: {text_lengths.mean():.0f} characters")
print(f"Max: {text_lengths.max()} characters")
print(f"Texts over 1000 chars: {(text_lengths > 1000).sum()}")

## 6. Perform Sentiment Analysis

Use the Cardiff NLP Twitter RoBERTa model to analyze sentiment of the merged comments.

In [ ]:
# Load the sentiment analysis pipeline
print("Loading sentiment analysis model...")
sentiment_pipeline = pipeline(
    "sentiment-analysis", 
    model="cardiffnlp/twitter-roberta-base-sentiment", 
    device=device
)
print("✅ Sentiment analysis model loaded!")

In [ ]:
# Apply sentiment analysis with proper truncation
print("Performing sentiment analysis...")
print("This may take a few minutes depending on the dataset size and device.")

try:
    # Convert to list and apply sentiment analysis
    results = sentiment_pipeline(
        grouped.tolist(), 
        truncation=True, 
        max_length=512, 
        batch_size=8
    )
    
    # Convert results to DataFrame
    sentiment_df = pd.DataFrame(results, index=grouped.index)
    sentiment_df.columns = ['merged_comment_sentiment', 'sentiment_score']
    
    # Map Cardiff NLP labels to meaningful sentiment names
    # LABEL_0 = Negative, LABEL_1 = Neutral, LABEL_2 = Positive
    label_mapping = {
        'LABEL_0': 'Negative',
        'LABEL_1': 'Neutral', 
        'LABEL_2': 'Positive'
    }
    
    sentiment_df['merged_comment_sentiment'] = sentiment_df['merged_comment_sentiment'].map(label_mapping)
    
    print("✅ Sentiment analysis completed!")
    print(f"Results shape: {sentiment_df.shape}")
    print("\nSentiment distribution:")
    print(sentiment_df['merged_comment_sentiment'].value_counts())
    print("\nFirst few results:")
    print(sentiment_df.head())
    
except Exception as e:
    print(f"❌ Error during sentiment analysis: {e}")
    print("This might be due to memory limitations or text length issues.")

In [ ]:
# Join with original posts data
print("Merging results with original data...")

# Get unique posts (drop duplicates)
post_df = df.drop_duplicates(subset='post_id').set_index('post_id')

# Join sentiment results
final_df = post_df.join(sentiment_df)

print("✅ Data merged successfully!")
print(f"Final dataset shape: {final_df.shape}")
print("\nColumns in final dataset:")
print(list(final_df.columns))
print("\nSample of final results:")
print(final_df[['merged_comment_sentiment', 'sentiment_score']].head())

## 7. Save and Download Results

Save the results to a CSV file and download it.

In [ ]:
# Save results to CSV
output_filename = "posts_with_sentiment.csv"
final_df.to_csv(output_filename, index=False)

print(f"✅ Results saved to {output_filename}")
print(f"File size: {os.path.getsize(output_filename) / 1024:.1f} KB")

# Download the file
from google.colab import files
files.download(output_filename)

print("\n🎉 Sentiment analysis complete!")
print(f"📊 Processed {len(final_df)} posts")
print(f"📁 Results saved and downloaded as '{output_filename}'")

## 8. Summary and Visualization (Optional)

Generate a quick summary of the sentiment analysis results.

In [ ]:
import matplotlib.pyplot as plt

# Create visualizations
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Sentiment distribution
sentiment_counts = final_df['merged_comment_sentiment'].value_counts()
ax1.pie(sentiment_counts.values, labels=sentiment_counts.index, autopct='%1.1f%%')
ax1.set_title('Sentiment Distribution')

# Sentiment scores distribution
ax2.hist(final_df['sentiment_score'], bins=20, alpha=0.7, edgecolor='black')
ax2.set_xlabel('Sentiment Score')
ax2.set_ylabel('Frequency')
ax2.set_title('Sentiment Score Distribution')

plt.tight_layout()
plt.show()

print("\n📈 Summary Statistics:")
print(f"Total posts analyzed: {len(final_df)}")
print(f"Average sentiment score: {final_df['sentiment_score'].mean():.3f}")
print(f"Most common sentiment: {final_df['merged_comment_sentiment'].mode()[0]}")
print("\nSentiment breakdown:")
for sentiment, count in sentiment_counts.items():
    percentage = (count / len(final_df)) * 100
    print(f"  {sentiment}: {count} posts ({percentage:.1f}%)")